In [64]:
import numpy as np
import pandas as pd
import mne
import math
import os

In [65]:
# loading windows
base_dir = r"C:\Users\jshin\OW_closedloopLIFU\eeg_windows\scope\sham"
theta_df = []
csv_files = [f for f in os.listdir(base_dir) if f.endswith(".csv")]
csv_files.sort(key=lambda f: os.path.getctime(os.path.join(base_dir, f)))

for idx, window in enumerate(csv_files):
    path = os.path.join(base_dir, csv_files[idx])
    idx_df = pd.read_csv(path) 
    theta_df.append(idx_df)
theta_df[1]

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel,offline z-score,median,mad,LIFU,t_rel,idx
0,898881.906570,16793.324219,0.451963,0.204270,0.849448,-0.351807,-0.415566,-0.108612,-0.415566,2.258929,-0.415566,0.0,-1.999942,1
1,898881.912747,16578.652344,0.498180,0.248183,0.850137,-0.351642,-0.415566,-0.559945,-0.415566,2.258929,-0.415566,0.0,-1.993765,1
2,898881.914084,16717.792969,0.533143,0.284241,0.843990,-0.353112,-0.415566,-0.956113,-0.415566,2.258929,-0.415566,0.0,-1.992428,1
3,898881.926774,16951.777344,0.555101,0.308137,0.828139,-0.356905,-0.415566,-1.102111,-0.415566,2.258929,-0.415566,0.0,-1.979739,1
4,898881.929936,16829.947266,0.562949,0.316912,0.800834,-0.363437,-0.415566,-1.137817,-0.415566,2.258929,-0.415566,0.0,-1.976577,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1734,898888.898993,10257.984375,8.814456,77.694626,29.068558,6.399177,8.559975,-1.277491,8.559975,1.587161,8.559975,0.0,4.992481,1
1735,898888.900984,10099.174805,9.717863,94.436867,30.943466,6.847720,8.559975,29.032837,8.559975,1.587161,8.559975,0.0,4.994472,1
1736,898888.901508,9891.559570,10.454575,109.298134,33.129108,7.370600,8.559975,20.959158,8.559975,1.587161,8.559975,0.0,4.994995,1
1737,898888.901746,10024.668945,11.011195,121.246422,35.546814,7.948999,8.559975,-12.991534,8.559975,1.587161,8.559975,0.0,4.995234,1


In [66]:
#when it sonicates
lifu_markers = pd.read_csv("lifu_markers_1_2back_sham.csv")
lifu_on_df = lifu_markers[lifu_markers["marker"] == "LIFU_ON"]
lifu_on = np.array(lifu_on_df["LSL_timestamp"])
lifu_on

array([898868.2512194, 898883.9065124, 898898.9166517, 898914.2355922,
       898929.6814601, 898966.3711552, 898986.8628563, 899002.5722446])

In [67]:
#finding baseline median and mean
medians = []
mads = []
tol = 0.002

for idx, event in enumerate(lifu_on):
    df = theta_df[idx]
    diff = np.abs(df["LSL Time"].values - event)
    i = np.argmin(diff)
    row = df.iloc[i]
    median = row["median"]
    mad = row["mad"]
    medians.append(median)
    mads.append(mad)
print(f"medians = {medians}")
print(f"mads = {mads}")

medians = [2.24307644367218, 1.6061542630195618, 0.8345083594322205, 0.8002529144287109, 0.5622989535331726, -0.060883268713951, -0.0014000535011291, 0.0552176237106322]
mads = [4.912068843841553, 1.5272916555404663, 5.726717948913574, 2.5626258850097656, 1.7324949502944946, 1.62848699092865, 1.5896050930023191, 1.617497205734253]


In [68]:
# median percent change calculation
idx = 1
pre_sonication= []
post_sonication= []
percent_change_df = pd.DataFrame(lifu_on, columns=["Time"])

#for loop
for idx, event in enumerate(lifu_on):
    df = theta_df[idx]
    pre_df = df[df["LSL Time"]<= event]
    pre_mean = np.mean(np.array(pre_df["Hold"].dropna()))
    post_df = df[df["LSL Time"]>= event]
    post_mean = np.mean(np.array(post_df["Hold"].dropna()))
    pre_sonication.append(pre_mean)
    post_sonication.append(post_mean)
percent_change_df["pre average median"] = pre_sonication
percent_change_df["post average median"] = post_sonication
# CHECK IF EQUATION IS CORRECT -- also slope
percent_change_df["percent change %"] = 100* (percent_change_df["post average median"] - percent_change_df["pre average median"])/ percent_change_df["pre average median"]
percent_change_df

,Time,pre average median,post average median,percent change %
0,898868.251219,13.303537,13.000941,-2.274558
1,898883.906512,0.032291,4.966596,15280.694350
2,898898.916652,2.110546,3.179073,50.628017
3,898914.235592,2.104729,1.315605,-37.492921
4,898929.681460,1.605593,0.248226,-84.539951
5,898966.371155,0.520925,1.725111,231.162742
6,898986.862856,0.258976,1.390692,436.995983
7,899002.572245,8.213673,0.684714,-91.663728


In [70]:
df = theta_df[1].copy()
time = lifu_on[1]
df = df[df['LSL Time'] > time]
median = df.iloc[np.argmin(np.abs(df["LSL Time"] - time))]['median']
# for each row, compare to median (median is more robust than mean)
threshold = 0.5 # do actual calculation based on mad
df = df[df['median'] < median +threshold] #<= threshold]
df


,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel,offline z-score,median,mad,LIFU,t_rel,idx
512,898883.918610,14447.976562,-3.745934,14.032023,9.018560,1.602526,1.527292,1.219979,1.527292,1.606154,1.527292,0.0,0.012097,1
513,898883.918990,14305.760742,-3.705849,13.733315,9.031431,1.605606,1.527292,1.583534,1.527292,1.606154,1.527292,0.0,0.012477,1
514,898883.920259,14555.432617,-3.610348,13.034612,9.014249,1.601495,1.527292,1.719502,1.527292,1.606154,1.527292,0.0,0.013747,1
515,898883.920845,14691.903320,-3.464018,11.999420,8.969839,1.590871,1.527292,1.662491,1.527292,1.606154,1.527292,0.0,0.014333,1
516,898883.921153,14468.528320,-3.272102,10.706650,8.903363,1.574967,1.527292,1.376792,1.527292,1.606154,1.527292,0.0,0.014641,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1734,898888.898993,10257.984375,8.814456,77.694626,29.068558,6.399177,8.559975,-1.277491,8.559975,1.587161,8.559975,0.0,4.992481,1
1735,898888.900984,10099.174805,9.717863,94.436867,30.943466,6.847720,8.559975,29.032837,8.559975,1.587161,8.559975,0.0,4.994472,1
1736,898888.901508,9891.559570,10.454575,109.298134,33.129108,7.370600,8.559975,20.959158,8.559975,1.587161,8.559975,0.0,4.994995,1
1737,898888.901746,10024.668945,11.011195,121.246422,35.546814,7.948999,8.559975,-12.991534,8.559975,1.587161,8.559975,0.0,4.995234,1


In [71]:
df_sorted = df.sort_values("LSL Time", ascending=False) # descending
MIN_SPACING = 1.0  # seconds
kept_rows = []
last_time = 0

for _, row in df_sorted.iterrows():
    t = row["LSL Time"]
    if np.abs(t - last_time) >= MIN_SPACING:
        kept_rows.append(row)
        last_time = t
    if len(kept_rows) == 15:
        break
drift_time = pd.DataFrame(kept_rows)
drift_time = np.array(drift_time['LSL Time'])
drift_time = np.sort(drift_time)
drift_time

array([898884.6013167, 898885.7031143, 898886.7147067, 898887.8721384,
       898888.9019707])